# Visualize a database of films

In [ ]:
# Visualisation d’une base de données de films

## Import Libraries

In [15]:
import os
import sys
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import scipy
import itertools
import collections
import httpx
import asyncio
import nest_asyncio
from tqdm.asyncio import tqdm_asyncio
from urllib.parse import urlparse

In [4]:
# Print current environment
print("--- Python Environment Details ---")
print(f"Python Version:      {sys.version}")
print(f"Executable Path:     {sys.executable}") # Crucial for knowing which Python/virtual env is running
print(f"Platform:            {sys.platform}")   # e.g., 'win32', 'linux', 'darwin'
print(f"Default Encoding:    {sys.getdefaultencoding()}")
print(f"Command Line Args:   {sys.argv}")

print("\n--- Python Module Search Path (sys.path) ---")
for path in sys.path:
    print(f" - {path}")

--- Python Environment Details ---
Python Version:      3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Executable Path:     /home/amiagarw/aiml01/bin/python3
Platform:            linux
Default Encoding:    utf-8
Command Line Args:   ['/home/amiagarw/aiml01/lib/python3.12/site-packages/ipykernel_launcher.py', '-f', '/home/amiagarw/.local/share/jupyter/runtime/kernel-27e6366b-2c7b-41c0-9f88-c7106f0e5570.json']

--- Python Module Search Path (sys.path) ---
 - /usr/lib/python312.zip
 - /usr/lib/python3.12
 - /usr/lib/python3.12/lib-dynload
 - 
 - /home/amiagarw/aiml01/lib/python3.12/site-packages


In [5]:
# Current Working directory
os.getcwd()

'/home/amiagarw/code/learning/challenge-hi-paris'

## Load Data

In [6]:
# Import data file
df_original = pd.read_csv(
    'data/movies.csv', 
    engine='python', 
    on_bad_lines='skip',
    header='infer',
    encoding='utf-8'
)

In [7]:
print(df_original.shape)

(9837, 9)


In [8]:
df_original.head()

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...


In [9]:
# Verify that we have the same number of columns
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9837 entries, 0 to 9836
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Release_Date       9837 non-null   object 
 1   Title              9828 non-null   object 
 2   Overview           9828 non-null   object 
 3   Popularity         9827 non-null   float64
 4   Vote_Count         9827 non-null   object 
 5   Vote_Average       9827 non-null   object 
 6   Original_Language  9827 non-null   object 
 7   Genre              9826 non-null   object 
 8   Poster_Url         9826 non-null   object 
dtypes: float64(1), object(8)
memory usage: 691.8+ KB


In [10]:
# Check for duplicate rows
# No duplicate rows found
duplicate_rows = df_original.duplicated()
print("Number of duplicate rows:", duplicate_rows.sum())

Number of duplicate rows: 0


## Download the images

In [16]:
# Patch the event loop to allow asyncio.run() in Jupyter
nest_asyncio.apply()

# Create a directory to save the posters if it doesn't exist
posters_dir = 'data/images'
os.makedirs(posters_dir, exist_ok=True)

async def download_poster_async(client, url, save_dir='data/images'):
    """
    Asynchronously downloads the poster from the URL and saves it.
    """
    if pd.isna(url) or not isinstance(url, str):
        return None
    
    try:
        # Extract the filename from the URL
        filename = os.path.basename(urlparse(url).path)
        if not filename:  
            filename = f"poster_{hash(url) % 100000}.jpg"
        
        filepath = os.path.join(save_dir, filename)
        
        # Download and save the image if it doesn't already exist
        if not os.path.exists(filepath):
            response = await client.get(url, timeout=30.0)
            response.raise_for_status()  
            with open(filepath, 'wb') as f:
                f.write(response.content)
        
        return filename
    
    except Exception as e:
        # print(f"Error downloading {url}: {e}") # Uncomment if you want to see individual errors
        return None

async def download_all_posters(urls, save_dir='data/images', max_concurrent=20):
    """
    Download all posters concurrently with a limit on concurrent connections.
    """
    os.makedirs(save_dir, exist_ok=True)
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def download_with_semaphore(client, url):
        async with semaphore:
            return await download_poster_async(client, url, save_dir)
    
    async with httpx.AsyncClient(timeout=30.0, limits=httpx.Limits(max_connections=100)) as client:
        tasks = [download_with_semaphore(client, url) for url in urls]
        results = await tqdm_asyncio.gather(*tasks, desc="Downloading posters")
    
    return results

# Run the async function (This will now work without throwing the RuntimeError)
print("Starting asynchronous download of posters...")
# poster_filenames = asyncio.run(download_all_posters(df_original['Poster_Url'].tolist(), posters_dir, max_concurrent=20))

# Add the filenames to the dataframe
# df_original['Poster_Filename'] = poster_filenames

print(f"Download complete! {sum(1 for f in poster_filenames if f is not None)} posters downloaded successfully.")

<string>:0: RuntimeWarning: coroutine 'download_all_posters' was never awaited


Starting asynchronous download of posters...


Download complete! 9826 posters downloaded successfully.


## Convert data types of columns

Change data type for columns. From above we saw that only Popularity was infered.

In [ ]:
# Create a "Shadow Dataframe" to preserve the raw, unconverted data
df_raw = df_original.copy()

# Apply conversions to df_original using 'coerce'

# DATES
# format='mixed' allows Pandas to infer the format for each row individually
# dayfirst=True to prefer DD-MM-YYYY when it's ambiguous
df_original['Release_Date'] = pd.to_datetime(
    df_original['Release_Date'], format='mixed', dayfirst=True, errors='coerce'
)

# NUMERICS (convert incoherent text like "N/A" or "1,000" to NaN)
df_original['Popularity'] = pd.to_numeric(df_original['Popularity'], errors='coerce').astype('float32')
df_original['Vote_Average'] = pd.to_numeric(df_original['Vote_Average'], errors='coerce').astype('float32')
df_original['Vote_Count'] = pd.to_numeric(df_original['Vote_Count'], errors='coerce').astype('Int32')

# STRINGS & CATEGORIES (type casting)
text_cols = ['Title', 'Overview', 'Genre', 'Poster_Url']
df_original[text_cols] = df_original[text_cols].astype('string')
df_original['Original_Language'] = df_original['Original_Language'].astype('category')


In [ ]:
# Splits the string into a list of genres
df_original['Genre_List'] = df_original['Genre'].str.split(', ')

## Data Cleaning

In [ ]:
# Isolate anomalies
# df_original.isna() will be True for BOTH originally missing values AND incoherent values that were coerced to NaN/NaT
anomaly_mask = df_original.isna().any(axis=1)

In [ ]:
# Filter the RAW dataframe to view the anomalies
# Filter df_raw instead of df_original, to see the ORIGINAL incoherent text 
# (e.g., seeing "unknown" or "1,000" instead of just <NA> or NaN
# Observation: We had 9837, but the count does not add up, implying wrong data types or missing info etc. Let us investigate.
# Since there are only 11 entries corresponding to these missing values, we can drop these (less than 1.2% of the samples)
df_anomalies = df_raw[anomaly_mask]
display(df_anomalies)
df_anomalies = df_anomalies.reset_index(drop=True)

In [ ]:
# Save for future, if information becomes available.
df_anomalies.to_csv('data/df_anomalies.csv', index=False, header=True)

In [ ]:
# Keep only the rows that are NOT anomalies
df_clean = df_original[~anomaly_mask].reset_index(drop=True)

# Delete the raw shadow dataframe
del df_raw

In [ ]:
# Verify it's clean
print(df_clean.isna().sum().sum()) # Should output 0

In [ ]:
# Verify the changes
print(df_clean.dtypes)

In [ ]:
df_clean.info()

In [ ]:
print(df_clean.shape)

In [ ]:
df_clean.head()

In [ ]:
# Save for future reuse.
df_clean.to_csv('data/df_clean.csv', index=False, header=True)

In [ ]:
# Find unique genere in the dataset (to be used later)
# Since Genre_List is a multi-label variable, standard encodings
# like Target, Frequency, or Binary Encoding will not work directly
# on the list as they expect a single string per row.
# Applying them would explode the dataframe, which duplicates rows.
# One-Hot Encoding can create high-dimensional, sparse data.
# but could still be used as we have only 19 individual genres and
# tree-based models (Random Forest, XGBoost) and Neural Networks 
# can handle 19 sparse binary columns effortlessly.
    
# Flatten the list of lists into a single set
unique_genres = set(itertools.chain.from_iterable(df_clean['Genre_List']))
print(sorted(unique_genres))

## Exploratory Data Analysis (EDA)

### 1. Univariate Analysis (Distributions & Outliers)

In [ ]:
# 1: Distribution of Continuous Metrics (Popularity, Vote_Average, Vote_Count)
# Analyzes skewness and identify extreme outliers. `Popularity` is likely heavily
# right-skewed, which may require log-transformation for future modeling.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df_clean['Popularity'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribution of Popularity (Right-Skewed)')
axes[0].set_xlabel('Popularity Score')

sns.histplot(df_clean['Vote_Average'], kde=True, ax=axes[1], color='lightgreen', binwidth=0.5)
axes[1].set_title('Distribution of Vote Average')
axes[1].set_xlabel('Rating (1-10)')

sns.boxplot(x=df_clean['Vote_Count'], ax=axes[2], color='salmon')
axes[2].set_title('Boxplot of Vote Count (Outliers)')
axes[2].set_xlabel('Number of Votes')
plt.tight_layout()
plt.show()


In [ ]:
## 2: Temporal Release Trends (Movies per Year)
## Identifies historical booms in film production and potential gaps in the dataset.
df_clean['Release_Year'] = df_clean['Release_Date'].dt.year
yearly_counts = df_clean['Release_Year'].value_counts().sort_index()

plt.figure(figsize=(12, 5))
sns.lineplot(x=yearly_counts.index, y=yearly_counts.values, marker='o', color='purple')
plt.title('Number of Movies Released Per Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 3: Seasonality of Releases (Movies per Month)
# Check if there are seasonal trends (e.g., summer blockbusters, holiday releases).

df_clean['Release_Month'] = df_clean['Release_Date'].dt.month
monthly_counts = df_clean['Release_Month'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
sns.barplot(x=monthly_counts.index, y=monthly_counts.values, palette='viridis', hue=monthly_counts.index, legend=False)
plt.title('Number of Movies Released by Month')
plt.xlabel('Month')
plt.ylabel('Count')
plt.xticks(ticks=range(12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.show()

### 2. Categorical & Multi-label Analysis (Genres & Languages)

In [ ]:
# 4: Individual Genre Frequency
# Since `Genre_List` is a multi-label column, explode it to find
# the most common individual genres.
genre_exploded = df_clean['Genre_List'].explode()
top_genres = genre_exploded.value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='magma', hue=top_genres.values, legend=False)
plt.title('Top 15 Most Common Individual Genres')
plt.xlabel('Count')
plt.ylabel('Genre')
plt.show()

In [ ]:
# 5: Most Common Genre Combinations**
# Analyze the raw `Genre` string to see which exact combinations
# are most popular (e.g., "Action, Adventure, Science Fiction").
top_combos = df_clean['Genre'].value_counts().head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_combos.values, y=top_combos.index, palette='cividis', hue=top_combos.values, legend=False)
plt.title('Top 10 Exact Genre Combinations')
plt.xlabel('Count')
plt.ylabel('Genre Combination')
plt.show()

In [ ]:
# 6: Language Distribution
## Visualize the dominance of English and the presence of other languages.
top_langs = df_clean['Original_Language'].value_counts().head(10)

plt.figure(figsize=(7, 7)) 
wedges, texts, autotexts = plt.pie(
    top_langs.values, 
    labels=top_langs.index, 
    autopct='%1.1f%%', 
    startangle=140, 
    pctdistance=1.1,      # Pushes the percentage text outside the pie wedge
    labeldistance=1.2     # Pushes the language labels further outside
)

# Format the text outside the pie
plt.setp(autotexts, size=9, weight="normal", color="black")
plt.setp(texts, size=10)

plt.title('Top 10 Original Languages by Movie Count')
plt.show()

In [ ]:
# 7: Genre Co-occurrence Analysis
# Which genres frequently appear together? (e.g., Does "Romance" often pair with "Comedy"?)
# Extract all unique pairs of genres from each movie
genre_pairs = df_clean['Genre_List'].apply(lambda x: list(itertools.combinations(sorted(x), 2)) if len(x) > 1 else [])
all_pairs = list(itertools.chain.from_iterable(genre_pairs))
pair_counts = pd.Series(collections.Counter(all_pairs)).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=pair_counts.values, y=[f"{p[0]} + {p[1]}" for p in pair_counts.index], palette='coolwarm', hue=pair_counts.values, legend=False)
plt.title('Top 10 Most Frequent Genre Pairs')
plt.xlabel('Frequency')
plt.ylabel('Genre Combination')
plt.show()

### 3. Bivariate & Multivariate Analysis (Relationships)

In [ ]:
# 8: Numerical Correlation Heatmap
# Check for multicollinearity or strong linear relationships between numerical features.
plt.figure(figsize=(6, 5))
corr_matrix = df_clean[['Popularity', 'Vote_Count', 'Vote_Average']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

In [ ]:
# 9: Confidence vs. Quality (Vote_Count vs. Vote_Average)**
# Do highly voted movies tend to have higher ratings, or is there a regression to the mean?
plt.figure(figsize=(10, 6))
sns.regplot(x='Vote_Count', y='Vote_Average', data=df_clean, scatter_kws={'alpha':0.1, 'color':'gray'}, line_kws={'color':'red'})
plt.title('Vote Count vs. Vote Average')
plt.xlabel('Vote Count')
plt.ylabel('Vote Average')
plt.show()

In [ ]:
# 10: Hype vs. Quality (Popularity vs. Vote_Average)
# Are popular movies actually well-received by critics/audiences?
plt.figure(figsize=(10, 6))
sns.regplot(x='Popularity', y='Vote_Average', data=df_clean, scatter_kws={'alpha':0.1, 'color':'blue'}, line_kws={'color':'orange'})
plt.title('Popularity vs. Vote Average')
plt.xlabel('Popularity Score')
plt.ylabel('Vote Average')
plt.show()

In [ ]:
# 11: Rating Distribution by Language
# Compare the distribution of ratings across the top languages
# to see if certain languages yield consistently higher/lower ratings.
top_5_langs = df_clean['Original_Language'].value_counts().head(5).index
df_lang_subset = df_clean[df_clean['Original_Language'].isin(top_5_langs)]

plt.figure(figsize=(10, 6))
sns.boxplot(x='Original_Language', y='Vote_Average', data=df_lang_subset, palette='Set2', legend=False)

plt.title('Distribution of Vote Average by Top 5 Languages')
plt.xlabel('Language')
plt.ylabel('Vote Average')
plt.show()

# Old: sns.boxplot(x='Original_Language', y='Vote_Average', data=df_lang_subset, palette='Set2')
# New:


In [ ]:
# 12: Genre Performance (Average Rating & Popularity per Genre)
# Which genres are critically acclaimed vs. which are commercially popular?
genre_stats = df_clean.explode('Genre_List').groupby('Genre_List')[['Vote_Average', 'Popularity']].mean()
genre_stats = genre_stats.sort_values('Vote_Average', ascending=True)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.barh(genre_stats.index, genre_stats['Vote_Average'], color='skyblue', label='Avg Vote Average')
ax1.set_xlabel('Avg Vote Average')
ax1.set_title('Average Rating and Popularity by Genre')

ax2 = ax1.twiny()
ax2.plot(genre_stats['Popularity'], genre_stats.index, color='red', marker='o', linestyle='-', label='Avg Popularity')
ax2.set_xlabel('Avg Popularity')

fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
plt.tight_layout()
plt.show()

### 4. Advanced Time-Series & Trend Analysis

In [ ]:
# 13: Evolution of Top Genres Over Time
#How has the film industry's focus shifted over the decades?
# (e.g., rise of Sci-Fi/Superhero movies).
top_5_genres = df_clean['Genre_List'].explode().value_counts().head(5).index
df_genres = df_clean.explode('Genre_List')
df_genres = df_genres[df_genres['Genre_List'].isin(top_5_genres)]
df_genres['Year'] = df_genres['Release_Date'].dt.year

genre_trend = df_genres.groupby(['Year', 'Genre_List']).size().unstack(fill_value=0)

plt.figure(figsize=(12, 6))
genre_trend.plot.area(stacked=True, alpha=0.7, colormap='tab10', ax=plt.gca())
plt.title('Evolution of Top 5 Genres Over Time')
plt.xlabel('Year')
plt.ylabel('Number of Movies')
plt.legend(title='Genre', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 14: Decade-wise Performance Comparison
# Compare the median ratings and vote counts across different decades.
df_clean['Decade'] = (df_clean['Release_Year'] // 10) * 10
valid_decades = df_clean[df_clean['Decade'] >= 1970] # Filter out very old/low data decades

plt.figure(figsize=(10, 6))
sns.boxplot(x='Decade', y='Vote_Average', hue='Decade', data=valid_decades, palette='mako', legend=False)
plt.title('Distribution of Vote Average by Decade')
plt.xlabel('Decade')
plt.ylabel('Vote Average')
plt.show()

In [ ]:
# 15: Monthly Popularity Trends
# Do movies released in specific months have higher average popularity scores?
monthly_pop = df_clean.groupby('Release_Month')['Popularity'].mean()

plt.figure(figsize=(10, 5))
sns.lineplot(x=monthly_pop.index, y=monthly_pop.values, marker='s', color='teal', linewidth=2.5)
plt.title('Average Popularity of Movies by Release Month')
plt.xlabel('Month')
plt.ylabel('Average Popularity')
plt.xticks(ticks=range(1, 13), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### 5. Text & Anomaly Analysis

In [ ]:
# 16: Overview Text Length vs. Engagement**
#Does a longer, more detailed synopsis correlate with higher popularity or more votes?
df_clean['Overview_Word_Count'] = df_clean['Overview'].str.split().str.len()

plt.figure(figsize=(12, 5))
sns.histplot(df_clean['Overview_Word_Count'], kde=True, color='brown')
plt.title('Distribution of Overview Length (Word Count)')
plt.xlabel('Number of Words')
plt.show()

# Check correlation
print("Correlation between Word Count and Popularity:", df_clean['Overview_Word_Count'].corr(df_clean['Popularity']))
print("Correlation between Word Count and Vote_Count:", df_clean['Overview_Word_Count'].corr(df_clean['Vote_Count']))

In [ ]:
# 17: Identifying "Hidden Gems" vs. "Overhyped Flops"**
# Use quantiles to segment movies into interesting quadrants for business insights.
pop_q75 = df_clean['Popularity'].quantile(0.75)
pop_q25 = df_clean['Popularity'].quantile(0.25)
vote_avg_q75 = df_clean['Vote_Average'].quantile(0.75)
vote_avg_q25 = df_clean['Vote_Average'].quantile(0.25)

hidden_gems = df_clean[(df_clean['Vote_Average'] >= vote_avg_q75) & (df_clean['Popularity'] <= pop_q25)]
overhyped = df_clean[(df_clean['Vote_Average'] <= vote_avg_q25) & (df_clean['Popularity'] >= pop_q75)]

print(f"--- Hidden Gems (High Rating, Low Popularity) --- Count: {len(hidden_gems)}")
display(hidden_gems[['Title', 'Vote_Average', 'Popularity']].head(5))

print(f"\n--- Overhyped Flops (Low Rating, High Popularity) --- Count: {len(overhyped)}")
display(overhyped[['Title', 'Vote_Average', 'Popularity']].head(5))

In [ ]:
# 18: Keyword Analysis in Overviews (High vs. Low Rated)
# Extract the most common meaningful words from highly-rated vs. low-rated movie overviews.
# (without NLP libraries)
def get_top_words(text_series, n=15):
    words = []
    stopwords = {'that', 'this', 'with', 'from', 'have', 'been', 'will', 'their', 'they', 'about', 'when', 'what', 'into', 'over', 'after'}
    for text in text_series.dropna():
        # Extract words with 4+ characters
        found = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
        words.extend([w for w in found if w not in stopwords])
    return collections.Counter(words).most_common(n)

high_rated_words = get_top_words(df_clean[df_clean['Vote_Average'] >= 8.0]['Overview'])
low_rated_words = get_top_words(df_clean[df_clean['Vote_Average'] <= 5.0]['Overview'])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh([w[0] for w in high_rated_words], [w[1] for w in high_rated_words], color='gold')
axes[0].set_title('Top Keywords in Highly Rated Movies (>= 8.0)')
axes[0].invert_yaxis()

axes[1].barh([w[0] for w in low_rated_words], [w[1] for w in low_rated_words], color='crimson')
axes[1].set_title('Top Keywords in Low Rated Movies (<= 5.0)')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

### 6. Hypothesis Tests

In [ ]:
# Sequel & Franchise Analysis (Binary Feature)
# Hypothesis: Movies with numbers or "Part" in the title (sequels) 
# might have different popularity/rating distributions than original films.
# Feature: Is_Sequel
# Identify potential sequels using Regex (digits, Roman numerals, or common sequel keywords)
sequel_pattern = r'(?i)(\d+|Part\s*[0-9]*|II|III|IV|V|Returns|Requiem|Resurrection|Reloaded)'
df_clean['Is_Sequel'] = df_clean['Title'].str.contains(sequel_pattern, regex=True).astype(int)

plt.figure(figsize=(8, 5))
sns.boxplot(x='Is_Sequel', y='Popularity', data=df_clean, palette='Set2', hue='Is_Sequel', legend=False)
plt.xticks(ticks=[0, 1], labels=['Original', 'Sequel/Franchise'])
plt.title('Popularity: Original vs. Sequel')
plt.show()

In [ ]:
# Release Day of the Week (Categorical Feature)
# Hypothesis: Movies released on weekends (especially Friday) might have higher initial popularity.
# Feature: Day_of_Week
df_clean['Day_of_Week'] = df_clean['Release_Date'].dt.day_name()

# Order days correctly
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_clean['Day_of_Week'] = pd.Categorical(df_clean['Day_of_Week'], categories=day_order, ordered=True)

plt.figure(figsize=(10, 5))
sns.barplot(x='Day_of_Week', y='Popularity', data=df_clean, estimator='median', errorbar=None, palette='viridis', hue='Day_of_Week')
plt.title('Median Popularity by Release Day')
plt.ylabel('Median Popularity')
plt.show()

In [ ]:
# Genre Count Impact (Numeric Feature)
# Hypothesis: Movies that are too generic (many genres) or too niche (one genre) might perform differently.
# Feature: Genre_Count

df_clean['Genre_Count'] = df_clean['Genre_List'].apply(len)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Genre_Count', y='Vote_Average', data=df_clean, ax=ax[0], palette='pastel', hue='Genre_Count', legend=False)
ax[0].set_title('Rating Distribution by Number of Genres')

sns.countplot(x='Genre_Count', data=df_clean, ax=ax[1], color='skyblue')
ax[1].set_title('Frequency of Genre Counts per Movie')

plt.tight_layout()
plt.show()

In [ ]:
# Title Length Analysis (Numeric Feature)
# Hypothesis: Short, punchy titles vs. long, descriptive titles might correlate with different metrics.
# Feature: Title_Length
df_clean['Title_Length'] = df_clean['Title'].str.len()

plt.figure(figsize=(8, 5))
sns.regplot(x='Title_Length', y='Popularity', data=df_clean, scatter_kws={'alpha':0.1, 's':5}, line_kws={'color':'red'})
plt.title('Popularity vs. Title Length')
plt.xlabel('Number of Characters in Title')
plt.show()

# Check for correlation
print(f"Correlation: {df_clean['Title_Length'].corr(df_clean['Popularity']):.3f}")

In [ ]:
# Target Variable Distribution (Defining the "Target")
# Classify movies as "Successful" or "Failed" (Classification)
# Feature: Is_Success

# Example: Defining Success as Top 25% Popularity
popularity_threshold = df_clean['Popularity'].quantile(0.75)
df_clean['Is_Hit'] = (df_clean['Popularity'] >= popularity_threshold).astype(int)

plt.figure(figsize=(8, 4))
sns.countplot(x='Is_Hit', data=df_clean, palette='coolwarm')
plt.xticks(ticks=[0, 1], labels=['Average/Flop', 'Hit'])
plt.title('Class Balance: Hit vs. Average (Top 25% Threshold)')
plt.show()


In [ ]:
# Multi-Hot Encoding Genres & Target Correlation
# Convert Genre_List into binary columns (Multi-Hot Encoding).
# This EDA step reveals which specific genres drive higher ratings.
# genre_dummies can be concatenated to the main dataframe for modeling.
# Create a Genre_Count feature (df_clean['Genre_List'].str.len()).

from sklearn.preprocessing import MultiLabelBinarizer

# 1. Multi-Hot Encode the genres
mlb = MultiLabelBinarizer()
genre_dummies = pd.DataFrame(mlb.fit_transform(df_clean['Genre_List']), 
                             columns=mlb.classes_, 
                             index=df_clean.index)

# 2. Combine with target and calculate average rating per genre
df_genre_target = pd.concat([genre_dummies, df_clean['Vote_Average']], axis=1)
genre_avg_rating = df_genre_target.groupby(level=0, axis=1).mean().mean().sort_values(ascending=False)

# 3. Visualize
plt.figure(figsize=(10, 6))
sns.barplot(x=genre_avg_rating.values, y=genre_avg_rating.index, 
            hue=genre_avg_rating.index, palette='viridis', legend=False)
plt.title('Average Vote_Average by Individual Genre')
plt.xlabel('Average Vote Average')
plt.ylabel('Genre')
plt.show()

In [ ]:
# Target Variable Distribution & Log Transformation
# Popularity or Vote_Count are heavily right-skewed.
# This EDA confirms whether a log transformation will normalize them for algorithms like Linear Regression or Ridge/Lasso.
# use Log_Popularity as the target variable or feature to satisfy the normality assumptions of certain models.
Log_Popularity = np.log1p(df_clean['Popularity'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original Distribution
sns.histplot(df_clean['Popularity'], kde=True, ax=axes[0], color='blue')
axes[0].set_title('Original Popularity (Highly Skewed)')

# Log-Transformed Distribution
sns.histplot(Log_Popularity, kde=True, ax=axes[1], color='green')
axes[1].set_title('Log-Transformed Popularity (Normalized)')

plt.tight_layout()
plt.show()

In [ ]:
# Temporal Feature Extraction (Weekend vs. Weekday)
# Release timing heavily impacts movie popularity.
# Extracting temporal features helps the model capture "weekend blockbuster" effects.
# Add Day_of_Week, Month, and Is_Weekend as categorical/numerical features. Extract Season (Winter, Spring, Summer, Fall).

# Extract temporal features
df_clean['Day_of_Week'] = df_clean['Release_Date'].dt.dayofweek
df_clean['Is_Weekend'] = df_clean['Day_of_Week'].isin([5, 6]).astype(int) # 5=Sat, 6=Sun

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Popularity by Day of Week
sns.boxplot(x='Day_of_Week', y='Popularity', data=df_clean, ax=axes[0], palette='Set2', hue='Day_of_Week')
axes[0].set_title('Popularity by Day of Week')
axes[0].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

# Popularity: Weekend vs Weekday
sns.boxplot(x='Is_Weekend', y='Popularity', data=df_clean, ax=axes[1], palette='Set1', hue='Is_Weekend')
axes[1].set_title('Popularity: Weekend vs Weekday Releases')
axes[1].set_xticklabels(['Weekday (0)', 'Weekend (1)'])

plt.tight_layout()
plt.show()

In [ ]:
# Outlier Detection & Capping Strategy
# Extreme outliers in Vote_Count or Popularity can severely skew regression models.
# This EDA helps you decide on a Winsorization (capping) strategy.
# Cap them using the IQR method or apply the log transformation before feeding them into tree-based or linear models.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw Outliers
sns.boxplot(x=df_clean['Vote_Count'], ax=axes[0], color='salmon')
axes[0].set_title('Vote_Count Outliers (Raw)')

# Log-scaled Outliers (Better visibility of the bulk of data)
sns.boxplot(x=np.log1p(df_clean['Vote_Count']), ax=axes[1], color='lightgreen')
axes[1].set_title('Log(Vote_Count) Outliers')

plt.tight_layout()
plt.show()

In [ ]:
# Multicollinearity Check with Engineered Features
# Before training, ensure engineered features (like genres) aren't 
# perfectly collinear with each other or existing numerical features.
# Combine numerical, engineered, and multi-hot encoded features
# For high correlation (e.g., > 0.8) between certain features,
# drop one of the correlated features or use regularization 
# (like Ridge/Lasso/ElasticNet) during modeling to prevent multicollinearity issues.

df_corr = pd.concat([
    df_clean[['Popularity', 'Vote_Count', 'Vote_Average']],
    genre_dummies,
    df_clean[['Is_Weekend']]
], axis=1)

plt.figure(figsize=(12, 10))
corr_matrix = df_corr.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix (Including Engineered Features)')
plt.show()

## Relationships Between Variables

### Relationship Analysis

In [ ]:
# Mann-Whitney U: If p < 0.05, we reject the null hypothesis and
# conclude that English movies receive a statistically significantly
# different (usually higher) number of votes compared to non-English movies.
# Kruskal-Wallis H: If p < 0.05, we reject the null hypothesis and
# conclude that the average rating (Vote_Average) is not the same across all genres.
# Follow up with a Dunn's post-hoc test to see exactly which genres differ.

# 1. Mann-Whitney U Test: Does Vote_Count differ between English and non-English movies?
english = df_clean[df_clean['Original_Language'] == 'en']['Vote_Count'].dropna()
non_english = df_clean[df_clean['Original_Language'] != 'en']['Vote_Count'].dropna()

stat, p_value = scipy.stats.mannwhitneyu(english, non_english, alternative='two-sided')
print(f"Mann-Whitney U Test (Vote_Count by Language):")
print(f"U-Statistic: {stat:.2f}, p-value: {p_value:.3e}")
print("Interpretation: A p-value < 0.05 indicates a statistically significant difference in Vote_Count between English and non-English movies.")

# 2. Kruskal-Wallis H Test: Does Vote_Average differ across the Top 5 Genres?
df_genre_exploded = df_clean.explode('Genre_List')
top_genres = df_genre_exploded['Genre_List'].value_counts().head(5).index
df_genre_top = df_genre_exploded[df_genre_exploded['Genre_List'].isin(top_genres)]

groups = [group['Vote_Average'].dropna() for name, group in df_genre_top.groupby('Genre_List')]
stat, p_value = scipy.stats.kruskal(*groups)
print(f"\nKruskal-Wallis H Test (Vote_Average by Top Genre):")
print(f"H-Statistic: {stat:.2f}, p-value: {p_value:.3e}")
print("Interpretation: A p-value < 0.05 indicates that at least one genre has a significantly different Vote_Average compared to the others (e.g., Documentaries might rate differently than Action movies).")

### Correlation Analysis

Calculate Pearson, Spearman, and Kendall Tau correlation coefficients to understand the relationships between numerical variables.

In [ ]:
# Pearson measures linear correlation. Spearman and Kendall measure monotonic correlation
# (how well the relationship can be described by a monotonic function).
# Because Popularity and Vote_Count are heavily skewed with outliers, Spearman and Kendall are more reliable.
# A high positive Spearman/Kendall coefficient indicates that as a movie's popularity increases, 
# the number of votes it receives consistently increases.

# Pearson vs. Spearman/Kendall:
# Because Popularity and Vote_Count have extreme outliers (blockbusters), the Pearson correlation might be skewed.
# Spearman and Kendall Tau are rank-based and robust to outliers, making them better
# indicators of the true monotonic relationship between popularity and vote count.
# A high positive coefficient (e.g., > 0.7) indicates a strong positive relationship: as popularity increases,
# the number of votes consistently increases.

pop = df_clean['Popularity'].dropna()
vc = df_clean['Vote_Count'].dropna()

# Pearson (Linear relationship)
pearson_corr, pearson_p = scipy.stats.pearsonr(pop, vc)
# Spearman (Monotonic relationship - robust to outliers)
spearman_corr, spearman_p = scipy.stats.spearmanr(pop, vc)
# Kendall Tau (Monotonic relationship - robust to outliers)
kendall_corr, kendall_p = scipy.stats.kendalltau(pop, vc)

print("Correlation: Popularity vs Vote_Count")
print(f"Pearson (Linear): {pearson_corr:.3f} (p-value: {pearson_p:.3e})")
print(f"Spearman (Monotonic): {spearman_corr:.3f} (p-value: {spearman_p:.3e})")
print(f"Kendall Tau: {kendall_corr:.3f} (p-value: {kendall_p:.3e})")


### Collinearity and Multicollinearity (VIF)

In [ ]:
# Multicollinearity Analysis (VIF)
# Variance Inflation Factor (VIF) to check for multicollinearity among numerical features.
# Multicollinearity can inflate the variance of the coefficient estimates and make 
# the estimates very sensitive to minor changes in the model.
# Consider dropping one of the correlated variables or using dimensionality reduction (like PCA).
# VIF = 1: No correlation.
# VIF 1 - 5: Moderate correlation.
# VIF > 5 (or > 10): High multicollinearity.

# If Popularity and Vote_Count have a VIF > 5, it indicates they are highly correlated.

# Select numerical features
X_vif = df_clean[['Popularity', 'Vote_Count', 'Vote_Average']].dropna().astype(float)
X_vif = sm.add_constant(X_vif) # Add constant for the intercept

vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]

print(vif_data)


## Feature Engineering

### Additional Columns

In [ ]:
# Release_Season: Useful for seasonal trend analysis
# (e.g., testing if Summer blockbusters have significantly higher popularity than Winter releases).
# Is_Franchise: Helps in understanding if sequels/franchises perform better than original movies.
# Title_Word_Count / Overview_Word_Count: Text-based features that can be used to see if longer
# titles or more detailed descriptions correlate with success (e.g., does a longer overview
# indicate a more complex movie that attracts a niche audience?).
# Success_Bucket: Converts the continuous Popularity variable into a categorical target variable,
# which is highly useful if you want to train a Classification Model
# (e.g., predicting if a movie will be a "Blockbuster").

# 1. Release Season (Categorical)
df_clean['Release_Season'] = df_clean['Release_Date'].dt.month.map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
})

# 2. Is Franchise / Sequel (Binary)
# Checks for numbers, "Part", "Saga", etc. in the title
df_clean['Is_Franchise'] = df_clean['Title'].str.contains(r'\b\d+\b|Part|Saga|Chronicles', regex=True, case=False).astype(int)

# 3. Text Length Features (Numerical)
df_clean['Title_Word_Count'] = df_clean['Title'].str.split().str.len()
df_clean['Overview_Word_Count'] = df_clean['Overview'].str.split().str.len()

# 4. Success Bucket (Categorical Target)
# Categorize movies into buckets based on Popularity quantiles
pop_q25 = df_clean['Popularity'].quantile(0.25)
pop_q50 = df_clean['Popularity'].quantile(0.50)
pop_q75 = df_clean['Popularity'].quantile(0.75)

df_clean['Success_Bucket'] = pd.cut(df_clean['Popularity'], 
                                    bins=[-1, pop_q25, pop_q50, pop_q75, float('inf')],
                                    labels=['Low', 'Medium', 'High', 'Blockbuster'])

print("New columns added: 'Release_Season', 'Is_Franchise', 'Title_Word_Count', 'Overview_Word_Count', 'Success_Bucket'")

### Standardize / Normalize Values

In [ ]:
# Standardization (StandardScaler):
# Transforms the data to have a mean of 0 and a standard deviation of 1.
# This is highly recommended for algorithms that assume the data is normally distributed 
# (e.g., Linear Regression, Logistic Regression) or algorithms that use gradient descent (e.g., Neural Networks).
# It is less sensitive to extreme outliers than MinMax scaling.
# Normalization (MinMaxScaler): Scales the data to a fixed range, usually [0, 1].
# This is highly recommended for algorithms that are sensitive to the magnitude of the
# data and do not assume a normal distribution (e.g., K-Nearest Neighbors, K-Means Clustering,
# Neural Networks with Sigmoid activation).
# Note: MinMax scaling is highly sensitive to outliers, so ensure handling outliers (e.g., via capping) before applying it.
    
from sklearn.preprocessing import StandardScaler, MinMaxScaler

print("\n--- Standardization and Normalization ---")

cols_to_scale = ['Popularity', 'Vote_Count', 'Vote_Average']

# 1. Standardization (Z-score Normalization)
scaler = StandardScaler()
df_clean[[f"{col}_Standardized" for col in cols_to_scale]] = scaler.fit_transform(df_clean[cols_to_scale])

# 2. Normalization (Min-Max Scaling)
min_max_scaler = MinMaxScaler()
df_clean[[f"{col}_MinMax" for col in cols_to_scale]] = min_max_scaler.fit_transform(df_clean[cols_to_scale])

print("Scaled columns added: '[Column]_Standardized' and '[Column]_MinMax'")
print(df_clean[cols_to_scale + ['Popularity_Standardized', 'Popularity_MinMax']].head())